# Atera WTA Breast Topology

This notebook is a thin reproducibility wrapper around the packaged `pyXenium` workflow for the Atera WTA FFPE breast Xenium sample.
It calls the published workflow entrypoint and then reads the fixed result bundle from `manuscript/atera_wta_breast_topology/`.


In [1]:
from pathlib import Path

dataset_root = Path(r"Y:\\long\\10X_datasets\\Xenium\\Atera\\WTA_Preview_FFPE_Breast_Cancer_outs")
tbc_results = dataset_root / r"sfplot_tbc_formal_wta\\results"
manuscript_root = Path(r"D:\\GitHub\\pyXenium\\manuscript")
output_dir = manuscript_root / "atera_wta_breast_topology"
cli_entrypoint = (
    "pyxenium atera-wta-breast-topology "
    f'"{dataset_root}" --manuscript-mode --manuscript-root "{manuscript_root}"'
)
python_entrypoint = "from pyXenium.validation import run_atera_wta_breast_topology"
print(f'dataset_root: {dataset_root}')
print(f'tbc_results: {tbc_results}')
print(f'output_dir: {output_dir}')
print(f'cli_entrypoint: {cli_entrypoint}')
print(f'python_entrypoint: {python_entrypoint}')


dataset_root: Y:\long\10X_datasets\Xenium\Atera\WTA_Preview_FFPE_Breast_Cancer_outs
tbc_results: Y:\long\10X_datasets\Xenium\Atera\WTA_Preview_FFPE_Breast_Cancer_outs\sfplot_tbc_formal_wta\results
output_dir: D:\GitHub\pyXenium\manuscript\atera_wta_breast_topology
cli_entrypoint: pyxenium atera-wta-breast-topology "Y:\long\10X_datasets\Xenium\Atera\WTA_Preview_FFPE_Breast_Cancer_outs" --manuscript-mode --manuscript-root "D:\GitHub\pyXenium\manuscript"
python_entrypoint: from pyXenium.validation import run_atera_wta_breast_topology


In [2]:
from pyXenium.validation import run_atera_wta_breast_topology

study = run_atera_wta_breast_topology(
    dataset_root=str(dataset_root),
    manuscript_mode=True,
    manuscript_root=str(manuscript_root),
)
print(f"artifact_dir: {study['artifact_dir']}")
print(f"runtime_seconds: {study['payload']['runtime_seconds']:.3f}")


artifact_dir: D:\GitHub\pyXenium\manuscript\atera_wta_breast_topology
runtime_seconds: 56.683


In [3]:
import json
import pandas as pd

summary = json.loads((output_dir / "summary.json").read_text(encoding="utf-8"))
print(pd.DataFrame(summary['cci_acceptance']).to_string(index=False))
print()
print(pd.DataFrame(summary['cci_pair_summaries'])[['ligand', 'receptor', 'best_sender_celltype', 'best_receiver_celltype', 'best_score']].to_string(index=False))


                                                                       check ligand receptor   observed_top_sender  pass  observed_rank observed_top_receiver
                              CSF1-CSF1R top sender should not be Mast Cells   CSF1    CSF1R CAFs, DCIS Associated  True            NaN                   NaN
CXCL12-CXCR4 should keep CAFs, DCIS Associated -> T Lymphocytes high-ranking CXCL12    CXCR4                   NaN  True            1.0                   NaN
                DLL4-NOTCH3 top hit should be Endothelial Cells -> Pericytes   DLL4   NOTCH3     Endothelial Cells  True            NaN             Pericytes

ligand receptor       best_sender_celltype           best_receiver_celltype  best_score
  CSF1    CSF1R      CAFs, DCIS Associated                      Macrophages    0.507387
CXCL12    CXCR4      CAFs, DCIS Associated                    T Lymphocytes    0.633882
 TGFB1   TGFBR2          Endothelial Cells                Endothelial Cells    0.529126
  JAG1   NOTCH1

In [4]:
print(pd.DataFrame(summary['pathway_acceptance']).to_string(index=False))
print()
print(pd.read_csv(output_dir / 'pathway_mode_comparison.csv').to_string(index=False))


                pathway             expected_best_celltypes            observed_best_celltype  pass
      MacrophageProgram                       [Macrophages]                       Macrophages  True
          PlasmaProgram                      [Plasma Cells]                      Plasma Cells  True
        VascularProgram      [Endothelial Cells, Pericytes]                 Endothelial Cells  True
       BasalDCISProgram  [Basal-like Structured DCIS Cells]  Basal-like Structured DCIS Cells  True
        ApocrineProgram                    [Apocrine Cells]                    Apocrine Cells  True
LuminalAmorphousProgram [Luminal-like Amorphous DCIS Cells] Luminal-like Amorphous DCIS Cells  True

                pathway       gene_topology_best_celltype  gene_topology_best_distance activity_point_cloud_best_celltype  activity_point_cloud_best_distance  retained_cell_count  retained_quantile activity_mode
        ApocrineProgram                    Apocrine Cells                     0.099613 

In [5]:
top_cci = pd.read_csv(output_dir / 'cci_sender_receiver_scores.csv').head(10)
print(top_cci[['ligand', 'receptor', 'sender_celltype', 'receiver_celltype', 'CCI_score']].to_string(index=False))


ligand receptor            sender_celltype                receiver_celltype  CCI_score
  DLL4   NOTCH3          Endothelial Cells                        Pericytes  0.662811
CXCL12    CXCR4      CAFs, DCIS Associated                    T Lymphocytes  0.633882
 TGFB1   TGFBR2          Endothelial Cells                Endothelial Cells  0.529126
  CSF1    CSF1R      CAFs, DCIS Associated                      Macrophages  0.507387
  JAG1   NOTCH1 11q13 Invasive Tumor Cells Basal-like Structured DCIS Cells  0.502909
 TGFB1   TGFBR2              T Lymphocytes                Endothelial Cells  0.485418
  JAG1   NOTCH1 11q13 Invasive Tumor Cells       11q13 Invasive Tumor Cells  0.474286
  DLL4   NOTCH3          Endothelial Cells                Endothelial Cells  0.472412
 TGFB1   TGFBR2  CAFs, Invasive Associated                Endothelial Cells  0.445751
 TGFB1   TGFBR2            Dendritic Cells                Endothelial Cells  0.424489


## Fixed figure outputs

Primary packaged figures generated by the workflow:

![CCI summary](../atera_wta_breast_topology/figures/cci_summary_heatmap.png)

![CCI hotspot](../atera_wta_breast_topology/figures/cci_hotspot_overlay.png)

![Pathway primary heatmap](../atera_wta_breast_topology/figures/pathway_to_cell_heatmap.png)

![Pathway hotspot](../atera_wta_breast_topology/figures/pathway_hotspot_overlay.png)
